# Arquitectura de Computadores — cuaderno integral

**Fuente:** apuntes de las clases del 12, 19, 20 y 26 de agosto de 2026, ejercicio de Huffman, nota del IBM PC 5150 y tarea del curso.

Este Colab convierte los apuntes en material ejecutable: teoría, ejemplos resueltos, calculadoras, verificaciones y ejercicios.

## Ruta de estudio
1. Contexto: IBM PC 5150
2. Huffman: algoritmo, árbol, codificación y calculadora
3. Representación de datos, BCD, IEEE 754 y circuitos eléctricos
4. Compuertas lógicas y álgebra booleana (sección final)

> Convenciones: se ignoran tildes cuando el ejercicio lo solicita; cada calculadora incluye ejemplos reproducibles. Las correcciones frente al apunte se señalan como **Nota de verificación**.

# 1. Codificación de Huffman

Huffman es un método de compresión **sin pérdida** que asigna códigos cortos a símbolos frecuentes y largos a símbolos raros. El código resultante es de **prefijo**: ningún código completo es prefijo de otro, por lo que se decodifica sin separadores.

## Algoritmo
1. Normalizar el texto según las reglas del ejercicio.
2. Contar frecuencias.
3. Crear un nodo por símbolo.
4. Repetir: extraer los dos nodos de menor frecuencia y unirlos.
5. Etiquetar ramas y recorrer el árbol hasta cada hoja.
6. Sustituir cada símbolo por su código.
7. Verificar que al decodificar se recupera el texto.

En el apunte: desempates en orden alfabético Z→A; se asigna `1` al nodo de menor frecuencia y `0` al de mayor frecuencia. La calculadora mantiene una regla determinista equivalente y muestra todos los pasos.

In [ ]:
from collections import Counter
import heapq, itertools, unicodedata

def normalizar(texto, contar_espacios=False, ignorar_tildes=True):
    if ignorar_tildes:
        texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = texto.upper()
    return texto if contar_espacios else texto.replace(' ', '')

def huffman(texto, contar_espacios=False):
    limpio = normalizar(texto, contar_espacios)
    frecuencias = Counter(limpio)
    contador = itertools.count()
    heap=[]
    # desempate reproducible: símbolo inverso para aproximar Z→A
    for simbolo, frecuencia in sorted(frecuencias.items(), reverse=True):
        heapq.heappush(heap, (frecuencia, next(contador), simbolo))
    pasos=[]
    if len(heap)==1:
        return limpio, frecuencias, {heap[0][2]:'0'}, [], heap[0][2]
    while len(heap)>1:
        f1, _, n1 = heapq.heappop(heap)
        f2, _, n2 = heapq.heappop(heap)
        nuevo=(n1,n2)  # izquierda=1 (menor/primero), derecha=0
        pasos.append((n1,f1,n2,f2,f1+f2))
        heapq.heappush(heap,(f1+f2,next(contador),nuevo))
    raiz=heap[0][2]; codigos={}
    def recorrer(nodo,prefijo=''):
        if isinstance(nodo,str): codigos[nodo]=prefijo or '0'; return
        recorrer(nodo[0],prefijo+'1'); recorrer(nodo[1],prefijo+'0')
    recorrer(raiz)
    return limpio, frecuencias, codigos, pasos, raiz

def codificar(texto,codigos): return ''.join(codigos[c] for c in texto)

def decodificar(bits,raiz):
    if isinstance(raiz,str): return raiz*len(bits)
    salida=[]; nodo=raiz
    for bit in bits:
        nodo=nodo[0] if bit=='1' else nodo[1]
        if isinstance(nodo,str): salida.append(nodo); nodo=raiz
    if nodo is not raiz: raise ValueError('Secuencia incompleta')
    return ''.join(salida)

### Ejercicio completo: “ARQUITECTURA DE COMPUTADORES”

Frecuencias anotadas sin espacios (26 símbolos): A3, C2, D2, E3, I1, M1, O2, P1, Q1, R3, S1, T3, U3. Las uniones del cuaderno conducen a un árbol de frecuencia total 26; la tabla y la cadena final estaban pendientes. La siguiente ejecución las completa y valida automáticamente.

> Puede existir más de un código Huffman óptimo cuando hay empates. Lo esencial es documentar la regla, mantenerla y verificar prefijo + decodificación.

In [ ]:
texto='ARQUITECTURA DE COMPUTADORES'
limpio, frecs, codigos, pasos, arbol = huffman(texto)
print('Frecuencias:', dict(sorted(frecs.items())))
print('\nUniones:')
for i,(a,fa,b,fb,total) in enumerate(pasos,1): print(f'{i:2}. {a}({fa}) + {b}({fb}) = {total}')
print('\nTabla de códigos:')
for s in sorted(codigos): print(f'{s}: {frecs[s]} → {codigos[s]}')
bits=codificar(limpio,codigos)
print('\nTexto normalizado:',limpio)
print('Bits:',bits)
print('Longitud:',len(bits),'bits; fijo de 4 bits:',len(limpio)*4,'bits')
print('Decodifica correctamente:',decodificar(bits,arbol)==limpio)

Frecuencias: {'A': 3, 'C': 2, 'D': 2, 'E': 3, 'I': 1, 'M': 1, 'O': 2, 'P': 1, 'Q': 1, 'R': 3, 'S': 1, 'T': 3, 'U': 3}

Uniones:
 1. S(1) + Q(1) = 2
 2. P(1) + M(1) = 2
 3. I(1) + O(2) = 3
 4. D(2) + C(2) = 4
 5. ('S', 'Q')(2) + ('P', 'M')(2) = 4
 6. U(3) + T(3) = 6
 7. R(3) + E(3) = 6
 8. A(3) + ('I', 'O')(3) = 6
 9. ('D', 'C')(4) + (('S', 'Q'), ('P', 'M'))(4) = 8
10. ('U', 'T')(6) + ('R', 'E')(6) = 12
11. ('A', ('I', 'O'))(6) + (('D', 'C'), (('S', 'Q'), ('P', 'M')))(8) = 14
12. (('U', 'T'), ('R', 'E'))(12) + (('A', ('I', 'O')), (('D', 'C'), (('S', 'Q'), ('P', 'M'))))(14) = 26

Tabla de códigos:
A: 3 → 011
C: 2 → 0010
D: 2 → 0011
E: 3 → 100
I: 1 → 0101
M: 1 → 00000
O: 2 → 0100
P: 1 → 00001
Q: 1 → 00010
R: 3 → 101
S: 1 → 00011
T: 3 → 110
U: 3 → 111

Texto normalizado: ARQUITECTURADECOMPUTADORES
Bits: 011101000101110101110100001011011110101100111000010010000000000011111100110011010010110000011
Longitud: 93 bits; fijo de 4 bits: 104 bits
Decodifica correctamente: True


In [ ]:
# Calculadora interactiva de Huffman
frase='INGRESA AQUI TU TEXTO'
limpio, frecs, codigos, pasos, arbol=huffman(frase, contar_espacios=False)
bits=codificar(limpio,codigos)
print('Códigos:',codigos)
print('Codificado:',bits)
print('Recuperado:',decodificar(bits,arbol))

Códigos: {'I': '111', 'E': '110', 'A': '101', 'X': '1001', 'S': '1000', 'R': '0111', 'Q': '0110', 'O': '0101', 'N': '0100', 'T': '001', 'G': '0001', 'U': '0000'}
Codificado: 111010000010111110100010110101100000111001000000111010010010101
Recuperado: INGRESAAQUITUTEXTO


### Ejercicios Huffman
1. Repite contando espacios y compara la tasa de compresión.
2. Cambia la regla de desempate y comprueba que la longitud total puede seguir siendo óptima.
3. Calcula `suma(frecuencia × longitud_del_código)`.
4. Explica por qué el código es de prefijo y demuestra la decodificación de cinco símbolos.

# 2. Representación de datos y circuitos digitales

## Sistemas de numeración
Un sistema posicional representa un número como suma de dígitos multiplicados por potencias de la base. Usamos decimal (10), binario (2), octal (8) y hexadecimal (16). ASCII asocia caracteres con números; por ejemplo, una palabra como UDEA se almacena como códigos binarios.

Ejemplos del apunte: 65 y 745 de decimal a binario; 879 a octal; 1352 a hexadecimal; además conversiones inversas desde binario y hexadecimal.

In [ ]:
def convertir_decimal(n,base):
    digitos='0123456789ABCDEF'
    if n==0:return '0'
    signo='-' if n<0 else ''; n=abs(n); salida=''
    while n: n,r=divmod(n,base); salida=digitos[r]+salida
    return signo+salida
for n,b in [(65,2),(745,2),(879,8),(1352,16)]:
    print(n,'→ base',b,':',convertir_decimal(n,b))
print('UDEA en ASCII:', ' '.join(f'{ord(c):08b}' for c in 'UDEA'))

65 → base 2 : 1000001
745 → base 2 : 1011101001
879 → base 8 : 1557
1352 → base 16 : 548
UDEA en ASCII: 01010101 01000100 01000101 01000001


## Aritmética binaria
Las reglas son las de decimal adaptadas a base 2: en suma `1+1=10`; en resta se pide prestado como `10₂`; multiplicar por 2 desplaza un bit a la izquierda.

**Nota de verificación:** para `1011₂ × 1011₂`, el resultado correcto es `1111001₂` (121 decimal). Si aparece otra cadena en el apunte, conviene comprobarla convirtiendo ambos lados a decimal.

In [ ]:
a=int('1011',2); b=int('1011',2)
print('Suma:',f'{a+b:b}')
print('Resta:',f'{a-b:b}')
print('Multiplicación:',f'{a*b:b}','=',a*b)

Suma: 10110
Resta: 0
Multiplicación: 1111001 = 121


## Enteros negativos
- **Signo-magnitud:** bit más significativo indica signo; tiene +0 y −0.
- **Complemento a 1:** invertir todos los bits; también tiene dos ceros.
- **Complemento a 2:** invertir y sumar 1; es el estándar porque simplifica la aritmética y tiene un solo cero.

En `n` bits, C2 cubre de `−2^(n−1)` a `2^(n−1)−1`. El apunte trabaja −13, −75, −76, −30, −48, −1863 en 12 bits y la suma de −420, −71 y −24 en 11 bits.

In [ ]:
def c2(n,bits):
    if not -(1<<(bits-1)) <= n < (1<<(bits-1)): raise OverflowError('No cabe')
    return f'{n & ((1<<bits)-1):0{bits}b}'
for n,b in [(-13,8),(-75,8),(-76,8),(-30,8),(-48,8),(-1863,12)]: print(n,c2(n,b))
nums=[-420,-71,-24]; total=sum(nums)
print('Suma en 11 bits:', [c2(x,11) for x in nums], '→',total,c2(total,11))

-13 11110011
-75 10110101
-76 10110100
-30 11100010
-48 11010000
-1863 100010111001
Suma en 11 bits: ['11001011100', '11110111001', '11111101000'] → -515 10111111101


## Punto flotante IEEE 754
En precisión simple: 1 bit de signo, 8 de exponente con sesgo 127 y 23 de fracción. Para convertir: pasa a binario, normaliza como `1.f × 2^e`, guarda `e+127` y completa la mantisa. Las fracciones como 0.3 o 0.16 suelen ser periódicas en binario, por lo que se redondean. El apunte desarrolla 196.3 y 658.16 y la conversión inversa.

In [ ]:
import struct
def ieee754_32(x):
    bits=''.join(f'{b:08b}' for b in struct.pack('>f',x))
    return {'signo':bits[0], 'exponente':bits[1:9], 'mantisa':bits[9:], 'completo':bits, 'hex':struct.pack('>f',x).hex().upper()}
for x in (196.3,658.16): print(x,ieee754_32(x))

196.3 {'signo': '0', 'exponente': '10000110', 'mantisa': '10001000100110011001101', 'completo': '01000011010001000100110011001101', 'hex': '43444CCD'}
658.16 {'signo': '0', 'exponente': '10001000', 'mantisa': '01001001000101000111101', 'completo': '01000100001001001000101000111101', 'hex': '44248A3D'}


## BCD 8421 y display de 7 segmentos
Cada dígito decimal se codifica por separado con pesos 8-4-2-1: 0=`0000`, …, 9=`1001`. Los patrones 1010–1111 no representan dígitos decimales válidos. Esto explica por qué BCD no es igual a convertir el número completo a binario. Un decodificador BCD–7 segmentos activa `a…g` para mostrar cada dígito.

In [ ]:
def a_bcd(n): return ' '.join(f'{int(d):04b}' for d in str(n))
segmentos={0:'abcdef',1:'bc',2:'abdeg',3:'abcdg',4:'bcfg',5:'acdfg',6:'acdefg',7:'abc',8:'abcdefg',9:'abcdfg'}
for n in (0,7,42,2026): print(n,'→',a_bcd(n))
print('Segmentos de 0 a 9:',segmentos)

0 → 0000
7 → 0111
42 → 0100 0010
2026 → 0010 0000 0010 0110
Segmentos de 0 a 9: {0: 'abcdef', 1: 'bc', 2: 'abdeg', 3: 'abcdg', 4: 'bcfg', 5: 'acdfg', 6: 'acdefg', 7: 'abc', 8: 'abcdefg', 9: 'abcdfg'}


## Laboratorio de circuitos eléctricos (20 de agosto)
Proceso: **calcular → diseñar → construir → medir**. Magnitudes: corriente (A), voltaje (V) y resistencia (Ω). Ley de Ohm: `V=IR`. En serie se suma resistencia; en paralelo se suman conductancias.

Instrumentos y seguridad: multímetro en paralelo para voltaje y en serie para corriente; seleccionar escala y bornes correctos; no medir resistencia con el circuito energizado. El montaje usa protoboard, resistores (código de colores), LED, jumpers, batería y eventualmente Arduino.

Ejemplo del apunte: fuente 9 V; R1=2.2 kΩ, R2=1.5 kΩ, R3=10 Ω. Total 3710 Ω, corriente ≈2.43 mA; caídas ≈5.34 V, 3.64 V y 0.02 V.

In [ ]:
def serie(voltaje,*resistencias):
    rt=sum(resistencias); i=voltaje/rt
    return rt,i,[i*r for r in resistencias]
rt,i,caidas=serie(9,2200,1500,10)
print(f'Rt={rt} Ω; I={i*1000:.3f} mA; caídas={[round(v,3) for v in caidas]}; suma={sum(caidas):.3f} V')
def paralelo(*resistencias): return 1/sum(1/r for r in resistencias)
print('Ejemplo paralelo 1kΩ || 2.2kΩ =',round(paralelo(1000,2200),2),'Ω')

Rt=3710 Ω; I=2.426 mA; caídas=[5.337, 3.639, 0.024]; suma=9.000 V
Ejemplo paralelo 1kΩ || 2.2kΩ = 687.5 Ω


# 3. Compuertas lógicas y álgebra booleana — sección final

- BUFFER: `Y=A`
- NOT: `Y=¬A`
- AND: `Y=A·B`
- OR: `Y=A+B`
- NAND: `Y=¬(A·B)`
- NOR: `Y=¬(A+B)`
- XOR: `Y=A⊕B` (entradas diferentes)
- XNOR: `Y=¬(A⊕B)` (entradas iguales)

NAND y NOR son universales: con una sola de ellas se pueden construir NOT, AND y OR, y por tanto cualquier función booleana.

In [ ]:
from itertools import product
def tabla_compuertas():
    print('A B | AND OR NAND NOR XOR XNOR')
    for A,B in product((0,1),repeat=2):
        AND=A&B; OR=A|B; XOR=A^B
        print(A,B,'| ',AND,' ',OR,'  ',1-AND,'  ',1-OR,' ',XOR,'  ',1-XOR)
tabla_compuertas()

A B | AND OR NAND NOR XOR XNOR
0 0 |  0   0    1    1   0    1
0 1 |  0   1    1    0   1    0
1 0 |  0   1    1    0   1    0
1 1 |  1   1    0    0   0    1


## Leyes fundamentales
- Identidad: `A+0=A`, `A·1=A`
- Dominación: `A+1=1`, `A·0=0`
- Idempotencia: `A+A=A`, `A·A=A`
- Complemento: `A+¬A=1`, `A·¬A=0`
- Involución: `¬¬A=A`
- Conmutativa y asociativa
- Distributiva: `A(B+C)=AB+AC` y `A+BC=(A+B)(A+C)`
- Absorción: `A+AB=A`, `A(A+B)=A`
- De Morgan: `¬(AB)=¬A+¬B`; `¬(A+B)=¬A·¬B`

### Ejemplos de simplificación
1. `A + A·B = A` (absorción).
2. `A·B + A·¬B = A(B+¬B)=A`.
3. `¬(A+B) = ¬A·¬B` (De Morgan).
4. Una XOR puede expresarse como `¬A·B + A·¬B`.

In [ ]:
# Verificación exhaustiva de identidades por tabla de verdad
def verificar(nombre,izq,der,n=2):
    ok=all(bool(izq(*x))==bool(der(*x)) for x in product((0,1),repeat=n))
    print(nombre,':',ok)
verificar('Absorción A+AB=A',lambda A,B:A or (A and B),lambda A,B:A)
verificar('De Morgan NOT(A AND B)',lambda A,B:not(A and B),lambda A,B:(not A) or (not B))
verificar('XOR',lambda A,B:A^B,lambda A,B:((not A) and B) or (A and (not B)))

Absorción A+AB=A : True
De Morgan NOT(A AND B) : True
XOR : True


## Aplicaciones y ejercicios finales
1. Diseña una alarma que se active si una puerta está abierta **o** hay humo.
2. Implementa XOR usando sólo NAND y verifica su tabla.
3. Simplifica `F=A·B + A·¬B + ¬A·B`.
4. Construye el decodificador de un dígito BCD a 7 segmentos y marca entradas inválidas.
5. Relaciona una función booleana con su circuito, tabla de verdad y aplicación real.

## Lista de control para estudiar
- Puedo explicar Huffman y verificar codificación/decodificación.
- Convierto entre bases y represento negativos en C2.
- Identifico signo, exponente y mantisa IEEE 754.
- Distingo BCD de binario y explico 7 segmentos.
- Aplico Ley de Ohm y uso seguro del multímetro.
- Construyo tablas de verdad y simplifico con leyes booleanas.

**Fuentes internas:** las siete notas del cuaderno de Evernote “Arquitectura de Computadores”. Revisar con la guía oficial del curso cuando haya una convención distinta de desempate, longitud de palabra o notación.

# Ampliación práctica: Huffman, BCD y circuitos

## Métricas de Huffman
Un código fijo usa `ceil(log2(m))` bits por símbolo; Huffman usa `suma(frecuencia × longitud)`. Verifica suma de frecuencias, propiedad de prefijo, decodificación exacta y ahorro frente al código fijo.

In [ ]:
import math
def metricas_huffman(texto):
    limpio, frecs, codigos, pasos, arbol=huffman(texto)
    total=len(limpio); bh=sum(frecs[s]*len(codigos[s]) for s in frecs)
    bf=total*max(1,math.ceil(math.log2(len(frecs))))
    prefijo=all(not codigos[a].startswith(codigos[b]) for a in codigos for b in codigos if a!=b)
    return {'bits_huffman':bh,'bits_fijos':bf,'longitud_media':bh/total,'ahorro_%':100*(1-bh/bf),'es_prefijo':prefijo}
print(metricas_huffman('ARQUITECTURA DE COMPUTADORES'))

{'bits_huffman': 93, 'bits_fijos': 104, 'longitud_media': 3.576923076923077, 'ahorro_%': 10.576923076923073, 'es_prefijo': True}


## BCD validado
Separa en grupos de cuatro bits. Sólo `0000`–`1001` son dígitos; `1010`–`1111` son inválidos. En una suma BCD, si un grupo supera 9 o produce acarreo, se suma `0110`. Documenta si el display es ánodo común (activo en 0) o cátodo común (activo en 1).

In [ ]:
def validar_bcd(cadena):
    x=cadena.replace(' ','')
    if len(x)%4 or any(c not in '01' for c in x): return False,'Formato inválido'
    grupos=[x[i:i+4] for i in range(0,len(x),4)]; malos=[g for g in grupos if int(g,2)>9]
    return (not malos, malos or [int(g,2) for g in grupos])
for x in ('0010 0000 0010 0110','1001 0101','1010 0001'): print(x,'→',validar_bcd(x))

0010 0000 0010 0110 → (True, [2, 0, 2, 6])
1001 0101 → (True, [9, 5])
1010 0001 → (False, ['1010'])


## Circuitos ampliados
**Serie:** misma corriente. **Paralelo:** mismo voltaje. **Mixto:** reduce bloques sucesivamente. Potencia: `P=VI=I²R=V²/R`. Para LED: `R=(Vfuente−VLED)/ILED`. Mide resistencia sin energía, voltaje en paralelo y corriente insertando el multímetro en serie; comienza por la escala más alta.

In [ ]:
def circuito_paralelo(v,*rs):
    req=paralelo(*rs); ramas=[v/r for r in rs]
    return {'Req_ohm':req,'I_total_A':v/req,'I_ramas_A':ramas,'potencias_W':[v*v/r for r in rs]}
def resistor_led(v,vl,i_mA):
    r=(v-vl)/(i_mA/1000); return {'R_min_ohm':r,'P_W':(i_mA/1000)**2*r}
print(circuito_paralelo(9,1000,2200))
print(resistor_led(9,2,15))

{'Req_ohm': 687.5, 'I_total_A': 0.01309090909090909, 'I_ramas_A': [0.009, 0.004090909090909091], 'potencias_W': [0.081, 0.03681818181818182]}
{'R_min_ohm': 466.6666666666667, 'P_W': 0.105}


# Ampliación final: compuertas lógicas y álgebra booleana

## Correspondencia completa con la clase del 26 de agosto
- BUFFER: conserva la entrada. NOT: la invierte.
- AND: 1 sólo si todas son 1. OR: 1 si al menos una es 1.
- NAND y NOR: negaciones de AND y OR; ambas son universales.
- XOR: 1 cuando las entradas son diferentes. XNOR: 1 cuando son iguales.

Las tablas de verdad siguientes verifican cada definición y permiten evaluar expresiones de dos y tres variables.

In [ ]:
def evaluar_compuertas(A,B):
    return {'BUFFER_A':A,'NOT_A':1-A,'AND':A&B,'OR':A|B,'NAND':1-(A&B),'NOR':1-(A|B),'XOR':A^B,'XNOR':1-(A^B)}
for A,B in product((0,1),repeat=2): print(A,B,evaluar_compuertas(A,B))

0 0 {'BUFFER_A': 0, 'NOT_A': 1, 'AND': 0, 'OR': 0, 'NAND': 1, 'NOR': 1, 'XOR': 0, 'XNOR': 1}
0 1 {'BUFFER_A': 0, 'NOT_A': 1, 'AND': 0, 'OR': 1, 'NAND': 1, 'NOR': 0, 'XOR': 1, 'XNOR': 0}
1 0 {'BUFFER_A': 1, 'NOT_A': 0, 'AND': 0, 'OR': 1, 'NAND': 1, 'NOR': 0, 'XOR': 1, 'XNOR': 0}
1 1 {'BUFFER_A': 1, 'NOT_A': 0, 'AND': 1, 'OR': 1, 'NAND': 0, 'NOR': 0, 'XOR': 0, 'XNOR': 1}


## NAND y NOR como compuertas universales

Con NAND: `NOT A = A NAND A`; `A AND B = NOT(A NAND B)`; `A OR B = (NOT A) NAND (NOT B)`.

Con NOR: `NOT A = A NOR A`; `A OR B = NOT(A NOR B)`; `A AND B = (NOT A) NOR (NOT B)`.

Esto demuestra que cualquier función construida con AND, OR y NOT puede implementarse usando exclusivamente NAND o exclusivamente NOR.

In [ ]:
def nand(a,b): return 1-(a&b)
def nor(a,b): return 1-(a|b)
def and_con_nand(a,b): return nand(nand(a,b),nand(a,b))
def or_con_nand(a,b): return nand(nand(a,a),nand(b,b))
def or_con_nor(a,b): return nor(nor(a,b),nor(a,b))
def and_con_nor(a,b): return nor(nor(a,a),nor(b,b))
for a,b in product((0,1),repeat=2):
    assert and_con_nand(a,b)==(a&b) and or_con_nand(a,b)==(a|b)
    assert and_con_nor(a,b)==(a&b) and or_con_nor(a,b)==(a|b)
print('Universalidad NAND y NOR verificada para todas las entradas')

Universalidad NAND y NOR verificada para todas las entradas


## Leyes booleanas de las notas

Identidad, dominación, idempotencia, complemento, doble negación, complemento de constantes, conmutativa, asociativa, distributiva, absorción y De Morgan.

Método de simplificación: elimina constantes; agrupa términos comunes; aplica complemento; usa absorción; mueve negaciones con De Morgan; confirma el resultado por tabla de verdad.

Ejemplos: `A+AB=A`; `AB+A¬B=A`; `¬(AB)=¬A+¬B`; `¬(A+B)=¬A¬B`; XOR=`¬AB+A¬B`.

In [ ]:
identidades={
 'A+0=A':(lambda A,B:A or 0,lambda A,B:A),
 'A·1=A':(lambda A,B:A and 1,lambda A,B:A),
 'A+AB=A':(lambda A,B:A or (A and B),lambda A,B:A),
 'AB+A¬B=A':(lambda A,B:(A and B) or (A and not B),lambda A,B:A),
 'De Morgan AND':(lambda A,B:not(A and B),lambda A,B:(not A) or (not B)),
 'De Morgan OR':(lambda A,B:not(A or B),lambda A,B:(not A) and (not B))}
for nombre,(izq,der) in identidades.items():
    assert all(bool(izq(A,B))==bool(der(A,B)) for A,B in product((0,1),repeat=2))
    print(nombre,'✓')

A+0=A ✓
A·1=A ✓
A+AB=A ✓
AB+A¬B=A ✓
De Morgan AND ✓
De Morgan OR ✓


## Aplicaciones resueltas

1. **Alarma:** `F=PuertaAbierta + Humo`; se implementa con OR.
2. **Luz de escalera:** dos interruptores activan la luz cuando difieren; XOR.
3. **Comparador de un bit:** la salida de igualdad es XNOR.
4. **Permiso seguro:** `F=Autorizado · PuertaCerrada`; AND.
5. **Control activo en bajo:** NAND/NOR permiten producir directamente señales negadas.

### Ejercicios
Construye la tabla de `F=(A+B)·¬C`; simplifica `AB+A¬B+¬AB`; implementa XOR sólo con NAND; implementa un comparador de dos bits; dibuja el circuito antes y después de simplificar.

In [ ]:
def tabla_3_variables(funcion):
    print('A B C | F')
    for A,B,C in product((0,1),repeat=3): print(A,B,C,'|',int(bool(funcion(A,B,C))))
tabla_3_variables(lambda A,B,C:(A or B) and not C)
# AB + A¬B + ¬AB = A + ¬A·B = A+B
for A,B in product((0,1),repeat=2):
    original=(A and B) or (A and not B) or ((not A) and B)
    assert bool(original)==bool(A or B)
print('Simplificación AB+A¬B+¬AB = A+B verificada')

A B C | F
0 0 0 | 0
0 0 1 | 0
0 1 0 | 1
0 1 1 | 0
1 0 0 | 1
1 0 1 | 0
1 1 0 | 1
1 1 1 | 0
Simplificación AB+A¬B+¬AB = A+B verificada


## Auditoría de cobertura del cuaderno

Quedaron incorporados los contenidos de las siete notas: clase del 12 (bases, ASCII, aritmética y Huffman), ejercicio completo de Huffman, clase del 19 (negativos, C1/C2, IEEE 754, BCD y siete segmentos), clase del 20 (laboratorio eléctrico y multímetro), clase del 26 (ocho compuertas, tablas, universalidad, leyes y simplificación), IBM PC 5150 y las convenciones de la tarea.

Las inconsistencias detectadas se señalan como notas de verificación; no se alteraron los apuntes originales de Evernote.

# Recursos visuales y ejercicios originales de las notas

Esta sección conserva las imágenes exportadas directamente desde Evernote. Las notas del 19 y 20 de agosto, Huffman, IBM PC 5150 y Tarea no contenían archivos de imagen exportables: sus recursos eran texto, tablas y grabaciones de audio, y sus ejercicios ya están reconstruidos arriba.

### Tabla de caracteres ASCII de la nota del 12 de agosto

![Tabla de caracteres ASCII de la nota del 12 de agosto](attachment:image.png)

Referencia visual original usada en clase. Para el ejercicio UDEA se emplean los códigos de U, D, E y A.

### Teoremas booleanos y ejercicios de simplificación

![Teoremas booleanos y ejercicios de simplificación](attachment:Teoremas booleanos.png)

Diapositiva original con OR, AND, NOT, De Morgan, conmutativa, distributiva y cinco expresiones prácticas.

### Ejercicios visibles en la diapositiva
1. `AB + A(B+A) + B(B+C)`
2. `[AB′(C+BD) + A′B′]C`
3. `A + AB + AB′C`
4. `ABCD + ABC′D + A′BC′D + A′BCD`
5. `[(AB+C′)(A+BC)]′`

Simplifica con idempotencia, absorción, complemento, distributiva y De Morgan; después verifica por tabla de verdad.

### Símbolos BUFFER y NOT

![Símbolos BUFFER y NOT](attachment:image.png)

BUFFER conserva A; NOT agrega la burbuja de inversión.

### Símbolos BUFFER y NOT — segunda imagen original

![Símbolos BUFFER y NOT — segunda imagen original](attachment:image (1).png)



### Símbolo AND

![Símbolo AND](attachment:image (2).png)

`Y=A·B`.

### Símbolo OR

![Símbolo OR](attachment:image (3).png)

`Y=A+B`.

### Símbolo NAND

![Símbolo NAND](attachment:image (4).png)

`Y=¬(A·B)`.

### Símbolo NOR

![Símbolo NOR](attachment:image (5).png)

`Y=¬(A+B)`.

### Símbolo XOR

![Símbolo XOR](attachment:image (6).png)

`Y=A⊕B`.

### Símbolo XNOR

![Símbolo XNOR](attachment:image (7).png)

`Y=¬(A⊕B)`.